# CheMari widgets in Jupyter

Every widget of [CheMari](https://github.com/N283T/chemari) in a Jupyter notebook. The widgets are [anywidget](https://anywidget.dev)s, so they display as they are, with no wrapper. The data are the tables in `results/precomputed/` (read from GitHub when
the notebook is not in a clone of the repository):

```bash
uv run jupyter lab examples/jupyter_widgets.ipynb
```

In [1]:
from pathlib import Path

import polars as pl

from chemari import (
    BitAtlas,
    BitImportance,
    ECFPMovie,
    ECFPStepper,
    MolGrid,
    MolPair,
    MolScatter,
    MorganBitTiles,
    MorganExplorer,
)
from chemari.examples import openadmet as bench

tables = bench.open_tables(Path("../results/precomputed"))
mols = (
    pl.read_parquet(tables["molecules"])
    .filter(pl.col("task") == "pxr/pEC50")
    .select("id", "smiles", pl.col("y").round(2).alias("pEC50"), "split")
)
train = mols.filter(pl.col("split") == "train")
test = mols.filter(pl.col("split") == "test")

## MolGrid

The selection is the widget's `selection` trait: read it in another cell, or watch it with `grid.observe(callback, names="selection")`.

In [12]:
grid = MolGrid(
    mols.sort("pEC50", descending=True),
    id_col="id",
    smiles_col="smiles",
    color_by="pEC50",
    group_by="split",
    page_size=12,
    selection_mode="pair",
)
grid

In [13]:
grid.selection

['OADMET-0001239', 'OADMET-0002337']

## MolPair

In [4]:
a, b = train.sort("pEC50", descending=True).head(2).iter_rows(named=True)
MolPair(a, b, value_cols=["pEC50"], show_common=True)

## MolScatter

In [5]:
pred = (
    pl.read_parquet(tables["predictions"])
    .filter(
        (pl.col("task") == "pxr/pEC50")
        & (pl.col("features") == "ECFP4 bit")
        & (pl.col("split") == "test")
    )
    .select("id", pl.col("pred").round(2).alias("predicted"))
)
MolScatter(
    test.join(pred, on="id").rename({"pEC50": "measured"}),
    x="measured",
    y="predicted",
    x_label="measured pEC50",
    y_label="predicted pEC50 (LightGBM on ECFP4 bits)",
    diagonal=True,
    same_axes=True,
    axis_fields=["measured", "predicted"],
)

## ECFPMovie

In [6]:
ECFPMovie()

## ECFPStepper

In [7]:
ECFPStepper("CC(=O)Oc1ccccc1C(=O)O")

## MorganBitTiles

In [8]:
MorganBitTiles(test["smiles"][0], reference=train["smiles"].to_list())

## MorganExplorer

In [9]:
pair = [
    {"id": r["id"], "smiles": r["smiles"], "label": f"pEC50 {r['pEC50']}"}
    for r in test.head(2).iter_rows(named=True)
]
MorganExplorer(
    pair,
    reference=train["smiles"].to_list(),
    y=train["pEC50"].to_numpy(),
    y_label="pEC50",
)

## BitAtlas

In [10]:
BitAtlas(train["smiles"].to_list(), ids=train["id"].to_list())

## BitImportance

In [11]:
gain = pl.read_parquet(tables["gain"]).filter(pl.col("task") == "pxr/pEC50").sort("bit")
BitImportance(
    train["smiles"].to_list(),
    importance={"gain": gain["gain"].to_numpy(), "mean |SHAP|": gain["shap_abs"].to_numpy()},
    ids=train["id"].to_list(),
    y=train["pEC50"].to_numpy(),
    y_label="pEC50",
)